In [2]:
import numpy as np
import pandas as pd
import datetime

import seaborn as sns
import matplotlib.pyplot as plt


from sklearn import model_selection, metrics
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier, plot_importance

from tensorflow import keras
from tensorflow.keras import layers

from eda_package.data import load_raw_data, clean_data

df = load_raw_data()
#df = clean_data(df)

In [ ]:
# columns with dtype object
categorical_features = list(df.select_dtypes(include = ['object']).columns)
categorical_features.remove('reservation_status_date')

# Label Encoder
label_encoder_feat = {}
for i, feature in enumerate(categorical_features):
    df[feature] = df[feature].astype(str)
    label_encoder_feat[feature] = LabelEncoder()
    df[feature] = label_encoder_feat[feature].fit_transform(df[feature])

In [ ]:
# Filling agent null values
df.agent.fillna(0, inplace = True)
df.agent = df.agent.astype(int)

In [5]:
# Select only numeric columns
numeric_df = df.select_dtypes(include = ["number"])

# Pearson correlation with target
corr_with_target = numeric_df.corr(method = 'pearson')["is_canceled"].sort_values(ascending = False)

# Display correlations
print("Correlation of numeric features with 'is_canceled':\n")
print(corr_with_target)

Correlation of numeric features with 'is_canceled':

is_canceled                       1.000000
deposit_type                      0.468634
lead_time                         0.293123
country                           0.264223
distribution_channel              0.167600
previous_cancellations            0.110133
adults                            0.060017
market_segment                    0.059338
days_in_waiting_list              0.054186
adr                               0.047557
stays_in_week_nights              0.024765
arrival_date_year                 0.016660
arrival_date_week_number          0.008148
children                          0.005048
arrival_date_month               -0.001491
stays_in_weekend_nights          -0.001791
arrival_date_day_of_month        -0.006130
meal                             -0.017678
company                          -0.020642
babies                           -0.032491
agent                            -0.046529
previous_bookings_not_canceled   -0.057358
r

In [7]:
# The features that are going to be used for the training and testing

df["total_guest"] = df["adults"] + df["children"] + df["babies"]

df_hotel = df[['hotel', 'lead_time', 'adults', 'children', 'babies', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest',
   'previous_cancellations', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
   'days_in_waiting_list', 'required_car_parking_spaces', 'total_of_special_requests', 'is_canceled', 'adr', 'total_guest']].copy()
df_hotel.head()

,hotel,lead_time,adults,children,babies,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,...,assigned_room_type,booking_changes,deposit_type,agent,days_in_waiting_list,required_car_parking_spaces,total_of_special_requests,is_canceled,adr,total_guest
0,1,342,2,0.0,0,135,3,1,0,0,...,2,3,0,0,0,0,0,0,0.0,2.0
1,1,737,2,0.0,0,135,3,1,0,0,...,2,4,0,0,0,0,0,0,0.0,2.0
2,1,7,1,0.0,0,59,3,1,0,0,...,2,0,0,0,0,0,0,0,75.0,1.0
3,1,13,1,0.0,0,59,2,0,0,0,...,0,0,0,304,0,0,0,0,75.0,1.0
4,1,14,2,0.0,0,59,6,3,0,0,...,0,0,0,240,0,0,1,0,98.0,2.0


In [8]:
# Initialize kfold column
df_hotel["kfold"] = -1

# Shuffle the dataset
df_hotel = df_hotel.sample(frac = 1, random_state = 42).reset_index(drop = True)

# Target values
target = df_hotel["is_canceled"].values

# Stratified K-Fold with 5 splits
kf = model_selection.StratifiedKFold(n_splits = 5, shuffle = False)

for fold, (train_idx, val_idx) in enumerate(kf.split(X = df_hotel, y = target)):
    df_hotel.loc[val_idx, "kfold"] = fold

# Verify
print(df_hotel.kfold.value_counts())

0    23878
1    23878
2    23878
3    23878
4    23878
Name: kfold, dtype: int64


In [ ]:
def run(fold, model_name):
    """
    Runs training and validation for a given fold using the specified model.
    """
    # Splitting training and validation data
    df_train = df_hotel[df_hotel.kfold != fold].reset_index(drop=True)
    df_valid = df_hotel[df_hotel.kfold == fold].reset_index(drop=True)

    # Features and target
    feature_cols = [col for col in df_hotel.columns if col not in ["is_canceled", "kfold"]]
    X_train = df_train[feature_cols].values
    y_train = df_train["is_canceled"].values

    X_valid = df_valid[feature_cols].values
    y_valid = df_valid["is_canceled"].values

    # Initializing model
    clf = models[model_name]

    # Training
    clf.fit(X_train, y_train)

    # Predictions
    preds = clf.predict(X_valid)

    # Evaluation
    acc = metrics.accuracy_score(y_valid, preds)
    f1_s = metrics.f1_score(y_valid, preds)
    print(f"Fold = {fold} | Accuracy = {acc:.4f} | F1 Score = {f1_s:.4f}")

    # Store fold metrics
    results.append({
        "fold": fold,
        "model": model_name,
        "accuracy": acc,
        "f1": metrics.f1_score(y_valid, preds),
        "precision": metrics.precision_score(y_valid, preds),
        "recall": metrics.recall_score(y_valid, preds)
    })

In [10]:
# Models Trained + DNN
models = {
    'logistic_regression' : LogisticRegression(C = 1.2),
    'xgboost': XGBClassifier(eta = 0.35, max_depth = 12),
    'random_forest' :  RandomForestClassifier(n_estimators = 200)
    }


In [ ]:
# Initializing results list before running folds
results = []

# XGBoost
for fold in range(5):
    run(fold, "xgboost")

# Evaluation
results_df = pd.DataFrame(results)
print("\nAll Fold-wise Metrics:")
print(results_df)
print("\nAverage Metrics Across Folds:")
print(results_df.drop(columns = ['fold']).groupby("model").mean())

Fold = 0 | Accuracy = 0.8837 | F1 Score = 0.8398
Fold = 1 | Accuracy = 0.8815 | F1 Score = 0.8353
Fold = 2 | Accuracy = 0.8742 | F1 Score = 0.8262
Fold = 3 | Accuracy = 0.8816 | F1 Score = 0.8364
Fold = 4 | Accuracy = 0.8775 | F1 Score = 0.8302

All Fold-wise Metrics:
   fold    model  accuracy        f1  precision    recall
0     0  xgboost  0.883659  0.839774   0.857076  0.823157
1     1  xgboost  0.881481  0.835274   0.860828  0.811193
2     2  xgboost  0.874194  0.826198   0.846072  0.807236
3     3  xgboost  0.881648  0.836420   0.856956  0.816846
4     4  xgboost  0.877502  0.830189   0.853222  0.808366

Average Metrics Across Folds:
         accuracy        f1  precision    recall
model                                           
xgboost  0.879697  0.833571   0.854831  0.813359


In [17]:
fold = 1

# Splitting training and validation data
df_train = df_hotel[df_hotel.kfold != fold].reset_index(drop=True)
df_valid = df_hotel[df_hotel.kfold == fold].reset_index(drop=True)

# Features and target
feature_cols = [col for col in df_hotel.columns if col not in ["is_canceled", "kfold"]]
X_train = df_train[feature_cols].values
y_train = df_train["is_canceled"].values

X_valid = df_valid[feature_cols].values
y_valid = df_valid["is_canceled"].values

# Initializing model
clf = XGBClassifier(eta = 0.35, max_depth = 12)

# Training
clf.fit(X_train, y_train)

# Predictions
preds = clf.predict(X_valid)

# Evaluation
acc = metrics.accuracy_score(y_valid, preds)
f1_s = metrics.f1_score(y_valid, preds)
print(f"Fold = {fold} | Accuracy = {acc:.4f} | F1 Score = {f1_s:.4f}")

# Store fold metrics
print({
    "accuracy": acc,
    "f1": metrics.f1_score(y_valid, preds),
    "precision": metrics.precision_score(y_valid, preds),
    "recall": metrics.recall_score(y_valid, preds)
})

Fold = 1 | Accuracy = 0.8815 | F1 Score = 0.8353
{'accuracy': 0.8814808610436385, 'f1': 0.8352735739231665, 'precision': 0.8608278344331134, 'recall': 0.8111927642736009}


In [21]:
from sklearn.metrics import classification_report
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

#{'gamma': 10, 'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 100}
final_params = {
    'booster' : 'gbtree',
    'n_estimators': 300, #100, 300
    'max_depth': 12, #3, 10
    'learning_rate': 0.2, #0.2, 0.05
    'gamma': 1, #10, 1
#    'lambda': 1,
#    'alpha': 0,
    'subsample': 0.5, #minimal impact
    'colsample_bytree': 0.3, #minimal impact
    'min_child_weight': 2, #minimal impact
    'random_state': 0,
    'scale_pos_weight': 3, #impact
    'eval_metric': 'logloss'
}
estimator3=XGBClassifier(**final_params)

estimator3.fit(X_train,y_train)
y_pre=estimator3.predict(X_valid)

print(f'Accuracy: {round(metrics.accuracy_score(y_valid, y_pre),2)}')
print(f'Recall: {round(metrics.recall_score(y_valid, y_pre),2)}')
print(f'Precision: {round(metrics.precision_score(y_valid, y_pre),2)}')
print(f'F1 score: {round(metrics.f1_score(y_valid, y_pre),2)}')
print(classification_report(y_valid,y_pre))

"""
Accuracy: 0.78
Recall: 0.68
Precision: 0.66
F1 score: 0.67
"""

Accuracy: 0.86
Recall: 0.9
Precision: 0.76
F1 score: 0.83
              precision    recall  f1-score   support

           0       0.93      0.84      0.88     15033
           1       0.76      0.90      0.83      8845

    accuracy                           0.86     23878
   macro avg       0.85      0.87      0.85     23878
weighted avg       0.87      0.86      0.86     23878



'\nAccuracy: 0.78\nRecall: 0.68\nPrecision: 0.66\nF1 score: 0.67\n'